In [1]:
import pandas as pd
import numpy as np

In [2]:
clinical_data = pd.DataFrame({
    "patient_id": ["P001", "P002", "P003", "P004", "P005", "P006",
                   "P007", "P008", "P009", "P010", "P011", "P012"],
    "centre": ["A", "A", "A", "A",
               "B", "B", "B", "B",
               "C", "C", "C", "C"],
    "age": [63, 55, 71, 49, 66, 58, 74, 52, 61, 57, 69, 64],
    "hpv_positive": [1, 0, 1, 1, 0, 1, 0, 1, 1, np.nan, 0, 1],
    "t_stage": [2, 3, 2, 1, 4, 2, 3, 2, 2, 3, np.nan, 4],
    "rfs_days": [800, 250, 1200, 430, 600, 310, 500, 900, 720, 280, 650, 400],
    "event": [0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1]
})

clinical_data

,patient_id,centre,age,hpv_positive,t_stage,rfs_days,event
0,P001,A,63,1.0,2.0,800,0
1,P002,A,55,0.0,3.0,250,1
2,P003,A,71,1.0,2.0,1200,0
3,P004,A,49,1.0,1.0,430,1
4,P005,B,66,0.0,4.0,600,1
5,P006,B,58,1.0,2.0,310,0
6,P007,B,74,0.0,3.0,500,1
7,P008,B,52,1.0,2.0,900,0
8,P009,C,61,1.0,2.0,720,0
9,P010,C,57,NaN,3.0,280,1


In [3]:
print(clinical_data.isna().sum())

patient_id      0
centre          0
age             0
hpv_positive    1
t_stage         1
rfs_days        0
event           0
dtype: int64


In [4]:
train_data = clinical_data[clinical_data["centre"] != "C"].copy()
test_data = clinical_data[clinical_data["centre"] == "C"].copy()

hpv_fill = train_data["hpv_positive"].mode()[0]
t_stage_fill = train_data["t_stage"].median()

train_data["hpv_positive"] = train_data["hpv_positive"].fillna(hpv_fill)
test_data["hpv_positive"] = test_data["hpv_positive"].fillna(hpv_fill)

train_data["t_stage"] = train_data["t_stage"].fillna(t_stage_fill)
test_data["t_stage"] = test_data["t_stage"].fillna(t_stage_fill)

print("Training missing values:")
print(train_data.isna().sum())

print("\nTest missing values:")
print(test_data.isna().sum())

Training missing values:
patient_id      0
centre          0
age             0
hpv_positive    0
t_stage         0
rfs_days        0
event           0
dtype: int64

Test missing values:
patient_id      0
centre          0
age             0
hpv_positive    0
t_stage         0
rfs_days        0
event           0
dtype: int64


In [5]:
from lifelines import CoxPHFitter

model_data = train_data[
    ["age", "hpv_positive", "t_stage", "rfs_days", "event"]
]

cph = CoxPHFitter()

cph.fit(
    model_data,
    duration_col="rfs_days",
    event_col="event"
)

cph.print_summary()

c:\Users\strangers\.conda\envs\hecktor-research\Lib\site-packages\lifelines\utils\__init__.py:1120: ConvergenceWarning: Column hpv_positive have very low variance when conditioned on death event present or not. This may harm convergence. This could be a form of 'complete separation'. For example, try the following code:

>>> events = df['event'].astype(bool)
>>> print(df.loc[events, 'hpv_positive'].var())
>>> print(df.loc[~events, 'hpv_positive'].var())

A very low variance means that the column hpv_positive completely determines whether a subject dies or not. See https://stats.stackexchange.com/questions/11109/how-to-deal-with-perfect-separation-in-logistic-regression.

  warnings.warn(dedent(warning_text), ConvergenceWarning)
c:\Users\strangers\.conda\envs\hecktor-research\Lib\site-packages\lifelines\fitters\coxph_fitter.py:1614: ConvergenceWarning: Newton-Raphson failed to converge sufficiently. Please see the following tips in the lifelines documentation: https://lifelines.readthed

<lifelines.CoxPHFitter: fitted with 8 total observations, 4 right-censored observations>
             duration col = 'rfs_days'
                event col = 'event'
      baseline estimation = breslow
   number of observations = 8
number of events observed = 4
   partial log-likelihood = -0.09
         time fit was run = 2026-10-04 19:59:55 UTC

---
               coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                      
age           -0.59      0.55      0.67           -1.91            0.72                0.15                2.06
hpv_positive -30.37      0.00     31.06          -91.24           30.50                0.00            1.76e+13
t_stage       -9.50      0.00     11.68          -32.39           13.38                0.00            6.50e+05

              cmp to     z    p  -log2(p)
covariate                                
age             0.00 -0.88 0.38      1.41
hpv_positive    0.00 -0.98 0.33      1.61
t_stage         0.00 -0.81 0.42      1.27
---
Concordance = 1.00
Partial AIC = 6.17
log-likelihood ratio test = 13.56 on 3 df
-log2(p) of ll-ratio test = 8.13

In [6]:
from lifelines.utils import concordance_index

test_model_data = test_data[
    ["age", "hpv_positive", "t_stage", "rfs_days", "event"]
]

test_risk = cph.predict_partial_hazard(
    test_model_data[["age", "hpv_positive", "t_stage"]]
)

test_c_index = concordance_index(
    test_model_data["rfs_days"],
    -test_risk,
    test_model_data["event"]
)

print("Held-out Centre C C-index:", round(test_c_index, 3))

Held-out Centre C C-index: 0.2


In [7]:
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index

results = []

for test_centre in clinical_data["centre"].unique():
    train = clinical_data[clinical_data["centre"] != test_centre].copy()
    test = clinical_data[clinical_data["centre"] == test_centre].copy()

    hpv_fill = train["hpv_positive"].mode()[0]
    t_stage_fill = train["t_stage"].median()

    train["hpv_positive"] = train["hpv_positive"].fillna(hpv_fill)
    test["hpv_positive"] = test["hpv_positive"].fillna(hpv_fill)

    train["t_stage"] = train["t_stage"].fillna(t_stage_fill)
    test["t_stage"] = test["t_stage"].fillna(t_stage_fill)

    features = ["age", "hpv_positive", "t_stage"]

    cph = CoxPHFitter()
    cph.fit(
        train[features + ["rfs_days", "event"]],
        duration_col="rfs_days",
        event_col="event"
    )

    risk = cph.predict_partial_hazard(test[features])

    c_index = concordance_index(
        test["rfs_days"],
        -risk,
        test["event"]
    )

    results.append({
        "held_out_centre": test_centre,
        "c_index": c_index
    })

results_df = pd.DataFrame(results)
results_df

c:\Users\strangers\.conda\envs\hecktor-research\Lib\site-packages\lifelines\utils\__init__.py:1120: ConvergenceWarning: Column t_stage have very low variance when conditioned on death event present or not. This may harm convergence. This could be a form of 'complete separation'. For example, try the following code:

>>> events = df['event'].astype(bool)
>>> print(df.loc[events, 't_stage'].var())
>>> print(df.loc[~events, 't_stage'].var())

A very low variance means that the column t_stage completely determines whether a subject dies or not. See https://stats.stackexchange.com/questions/11109/how-to-deal-with-perfect-separation-in-logistic-regression.

  warnings.warn(dedent(warning_text), ConvergenceWarning)
c:\Users\strangers\.conda\envs\hecktor-research\Lib\site-packages\lifelines\fitters\coxph_fitter.py:1614: ConvergenceWarning: Newton-Raphson failed to converge sufficiently. Please see the following tips in the lifelines documentation: https://lifelines.readthedocs.io/en/latest/Exa

,held_out_centre,c_index
0,A,0.200000
1,B,0.333333
2,C,0.200000


In [8]:
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index

results = []

for test_centre in clinical_data["centre"].unique():

    train = clinical_data[clinical_data["centre"] != test_centre].copy()
    test = clinical_data[clinical_data["centre"] == test_centre].copy()

    # Learn missing-value replacements from training data only
    hpv_fill = train["hpv_positive"].mode()[0]
    t_stage_fill = train["t_stage"].median()

    train["hpv_positive"] = train["hpv_positive"].fillna(hpv_fill)
    test["hpv_positive"] = test["hpv_positive"].fillna(hpv_fill)

    train["t_stage"] = train["t_stage"].fillna(t_stage_fill)
    test["t_stage"] = test["t_stage"].fillna(t_stage_fill)

    features = ["age", "hpv_positive", "t_stage"]

    # Small penalty makes the model more stable
    cph = CoxPHFitter(penalizer=0.1)

    cph.fit(
        train[features + ["rfs_days", "event"]],
        duration_col="rfs_days",
        event_col="event"
    )

    risk = cph.predict_partial_hazard(test[features])

    c_index = concordance_index(
        test["rfs_days"],
        -risk,
        test["event"]
    )

    results.append({
        "held_out_centre": test_centre,
        "c_index": round(c_index, 3)
    })

results_df = pd.DataFrame(results)

results_df

,held_out_centre,c_index
0,A,0.200
1,B,0.333
2,C,0.200


In [9]:
mean_c = results_df["c_index"].mean()
std_c = results_df["c_index"].std()
min_c = results_df["c_index"].min()
max_c = results_df["c_index"].max()

print("Mean C-index:", round(mean_c, 3))
print("Standard deviation:", round(std_c, 3))
print("Minimum C-index:", round(min_c, 3))
print("Maximum C-index:", round(max_c, 3))

Mean C-index: 0.244
Standard deviation: 0.077
Minimum C-index: 0.2
Maximum C-index: 0.333


In [10]:
import sys
sys.path.append("../src")

from clinical_pipeline import split_by_centre, fill_missing_values

In [11]:
train_check, test_check = split_by_centre(clinical_data, "C")

train_check, test_check = fill_missing_values(train_check, test_check)

print("Train centres:", train_check["centre"].unique())
print("Test centre:", test_check["centre"].unique())

print("\nTrain missing values:")
print(train_check.isna().sum())

print("\nTest missing values:")
print(test_check.isna().sum())

Train centres: ['A' 'B']
Test centre: ['C']

Train missing values:
patient_id      0
centre          0
age             0
hpv_positive    0
t_stage         0
rfs_days        0
event           0
dtype: int64

Test missing values:
patient_id      0
centre          0
age             0
hpv_positive    0
t_stage         0
rfs_days        0
event           0
dtype: int64


In [12]:
from survival_model import train_cox_model, evaluate_cox_model

In [13]:
features = ["age", "hpv_positive", "t_stage"]

train_check, test_check = split_by_centre(clinical_data, "C")
train_check, test_check = fill_missing_values(train_check, test_check)

model = train_cox_model(train_check, features)

c_index = evaluate_cox_model(
    model,
    test_check,
    features
)

print("Held-out Centre C C-index:", round(c_index, 3))

Held-out Centre C C-index: 0.2


In [15]:
from survival_model import leave_one_centre_out_evaluation

ImportError: cannot import name 'leave_one_centre_out_evaluation' from 'survival_model' (e:\Hecktor-centre-aware-prognosis\notebooks\../src\survival_model.py)

In [16]:
import importlib
import survival_model

importlib.reload(survival_model)

from survival_model import leave_one_centre_out_evaluation

print("Updated survival_model loaded successfully.")

Updated survival_model loaded successfully.


In [18]:
features = ["age", "hpv_positive", "t_stage"]

full_results = leave_one_centre_out_evaluation(
    clinical_data,
    features
)

full_results

,held_out_centre,c_index
0,A,0.200
1,B,0.333
2,C,0.200
